# Build mobility-ATUS distance matrix

Outcomes:
- Compute sequence metrics (number of activities, turnover, reciprocity, entropy, time use, transitions) for ATUS respondents and mobility user-days
- Normalize the metrics so features are on comparable scales
- Compute the cosine distance matrix between mobility user-days (rows) and ATUS respondents (columns)

This example notebook follows `walkthrough_03_process_atus.ipynb` (ATUS sequences + clusters) and precedes `walkthrough_05_run_calibration.ipynb` (k-NN cluster assignment on this matrix).

**Important: This example uses the synthetic mobility data in `data/mobility/synthetic_mobility_38060.csv`. However, `walkthrough_05` loads a precomputed matrix built the same way from real mobility data, which cannot be shared.**

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

%load_ext autoreload
%autoreload 2
from mobcalibrate.preprocessing import (
    compute_metrics_for_all_sequences,
    normalize_sequence_metrics,
    metrics_cosine_D,
)

### Configs (START HERE)

In [ ]:
# ======================
# PATHS
# ======================
CBSA_CODE = 38060
DATA_DIR = "data/processed"

# Inputs
ATUS_SEQ_FILE  = f"{DATA_DIR}/atus_seq_{CBSA_CODE}.csv"    # from walkthrough_03 (intermediate location codes 1-8)
ATUS_META_FILE = f"{DATA_DIR}/atus_meta_{CBSA_CODE}.csv"   # from walkthrough_03
MOB_SEQ_FILE   = "data/mobility/synthetic_mobility_38060.csv"     # user-day sequences (one column per T-minute cell) + GEOID

# Output
OUT_PATH_DIST = f"{DATA_DIR}/distance_matrix_synthetic_{CBSA_CODE}.parquet"

# ======================
# ALPHABET (same as walkthrough_03)
# ======================
# intermediate ATUS location code -> final alphabet
COLLAPSE_MAP = defaultdict(lambda: 'Unspecified place', {
    1: 'Home',
    2: 'Work',
    3: 'Unspecified place',
    4: 'POI',   # restaurants/bars
    5: 'POI',   # stores/mall
    6: 'POI',   # school
    7: 'POI',   # outdoors
    8: 'POI',   # other poi
})
ALL_LABELS = ['Home', 'Work', 'Unspecified place', 'POI']

# ======================
# PARAMETERS
# ======================
T = 30  # sequence cell width in minutes
MOB_SEQUENCE_COLS = [str(n) for n in range(int(24 * 60 / T))]

NORMALIZE = True  # rescale metrics; must match the setting used to cluster ATUS in walkthrough_03

## 1. ATUS sequence metrics

The ATUS sequences saved by `walkthrough_03` hold the intermediate location codes (1-8). They are collapsed to the final alphabet before computing the metrics.

In [ ]:
atus_meta = pd.read_csv(ATUS_META_FILE)
atus_seq  = pd.read_csv(ATUS_SEQ_FILE, index_col=0)

# rows must follow atus_meta's respondent order (cluster labels are matched by position downstream)
assert (atus_seq.index.values == atus_meta['TUCASEID'].values).all()

# collapse intermediate location codes -> final alphabet
atus_labels = atus_seq.map(lambda x: COLLAPSE_MAP[x])
atus_labels.head()

In [ ]:
atus_metrics = compute_metrics_for_all_sequences(atus_labels.values, all_labels=ALL_LABELS)
atus_metrics.index = atus_labels.index

if NORMALIZE:
    atus_metrics = normalize_sequence_metrics(atus_metrics, ALL_LABELS)

atus_metrics.head()

## 2. Mobility sequence metrics

The same metrics, computed for each mobility user-day. The sequence cells must use exactly the labels in `ALL_LABELS`.

In [ ]:
# read GEOID as a string to keep its leading zero
mob_seq = pd.read_csv(MOB_SEQ_FILE, index_col=0, dtype={'GEOID': str})
mob_seq.index.name = 'user_id'   # synthetic ids from the file's row index
mob_seq.head()

In [ ]:
mob_metrics = compute_metrics_for_all_sequences(mob_seq[MOB_SEQUENCE_COLS].values, all_labels=ALL_LABELS)
mob_metrics.index = mob_seq.index

if NORMALIZE:
    mob_metrics = normalize_sequence_metrics(mob_metrics, ALL_LABELS)

mob_metrics.head()

## 3. Cosine distance matrix

Both metric tables have the same columns in the same order (every ordered pair of labels has an `edge_(a, b)` column, zero if unobserved), so they can be compared directly. Rows are mobility user-days and columns are ATUS respondents, in `atus_meta` order. A `GEOID` column carries each user's home CBG, which `walkthrough_05` uses to match users to the ACS data.

The same steps are wrapped by `helpers.mobility.distance_to_atus`, which the quickstart notebooks use.

In [ ]:
# same features in the same order on both sides
assert list(mob_metrics.columns) == list(atus_metrics.columns), 'metric columns differ: check the sequence labels'

D = metrics_cosine_D(mob_metrics, atus_metrics)

dist_df = pd.DataFrame(D, index=mob_metrics.index, columns=atus_metrics.index.astype(str))
dist_df['GEOID'] = mob_seq['GEOID'].values

print('dist_df shape:', dist_df.shape)
dist_df.head()

In [ ]:
# save for walkthrough_05 (point its DIST_MATRIX_FILE here to use it)
# dist_df.to_parquet(OUT_PATH_DIST)